# Notebook 1 — AIS Input, Cleaning, Spatial Validation, and Visual Inspection

Notebook ini dibangun ulang berdasarkan struktur sebenarnya dari `2026 maritim.ais.csv`.

Kolom sumber utama:
- waktu: `created_at`
- identitas kapal: `mmsi`
- posisi: `lat`, `lon`
- kecepatan: `sog`
- arah: `cog`
- status data: `valid`
- status navigasi: `navstatus`

Output Stage 1:
1. `01_ais_clean.csv`
2. `01_ais_rejected.csv`
3. `01_cleaning_summary.csv`
4. `01_rejection_reason_summary.csv`
5. `01_vessel_summary.csv`
6. `01_spatial_validation_summary.csv`
7. `01_trajectory_segment_summary.csv`
8. `01_ais_full_validation_map.html`
9. peta inspeksi terpilih berdasarkan MMSI, tanggal, dan jam.


In [ ]:
#@title Shared MFAR paths and stage initialization
from pathlib import Path
from datetime import datetime, timezone
import os, sys, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

# Locate the code repository only; all simulation I/O paths are resolved by
# src.mfar_paths through MFAR_GDRIVE_ROOT or the mounted/synchronized Drive.
_code_candidates = [Path.cwd(), Path.cwd().parent]
if os.environ.get("MFAR_CODE_ROOT"):
    _code_candidates.insert(0, Path(os.environ["MFAR_CODE_ROOT"]))
_code_candidates.extend([
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
    Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/MFAR_Modular_Colab_Pipeline"),
])
for _candidate in _code_candidates:
    if (_candidate / "src" / "mfar_paths.py").is_file():
        sys.path.insert(0, str(_candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "Modul src/mfar_paths.py tidak ditemukan. Jalankan notebook dari repository "
        "atau tetapkan MFAR_CODE_ROOT ke folder repository."
    )

from src.mfar_paths import (
    AIS_RAW_PATH, VEHICLE_ARRIVAL_PATH, DATA_RAW_DIR, CONFIG_DIR,
    STAGE_OUTPUT_DIR, STAGE_01_DIR, STAGE_02_DIR, STAGE_03_DIR,
    STAGE_04_DIR, STAGE_05_DIR, STAGE_06_DIR, STAGE_07_DIR,
    validate_csv_input, validate_raw_inputs, validate_writable_directory,
    write_execution_metadata,
)

_MFAR_STARTED_AT = datetime.now(timezone.utc)

NOTEBOOK_NAME = "01_AIS_Input_and_Cleaning.ipynb"
RAW, CONFIG, STAGE = DATA_RAW_DIR, CONFIG_DIR, STAGE_01_DIR
AIS_FILE = AIS_RAW_PATH
validate_raw_inputs(NOTEBOOK_NAME)
validate_writable_directory(STAGE, NOTEBOOK_NAME, 1)
print("AIS source:", AIS_FILE)
print("Output folder:", STAGE)


In [ ]:
#@title Read and verify actual AIS columns
raw = validate_csv_input(AIS_FILE, required_source_columns if 'required_source_columns' in globals() else ["created_at","mmsi","lat","lon","sog","cog","valid","navstatus"], NOTEBOOK_NAME, 1)

print("Jumlah baris mentah:", len(raw))
print("Jumlah kolom:", len(raw.columns))
print("Nama kolom:")
print(raw.columns.tolist())

required_source_columns = [
    "created_at", "mmsi", "lat", "lon",
    "sog", "cog", "valid", "navstatus"
]

missing_source_columns = [
    c for c in required_source_columns
    if c not in raw.columns
]

if missing_source_columns:
    raise ValueError(
        "Kolom wajib tidak ditemukan: "
        + ", ".join(missing_source_columns)
    )

display(raw[required_source_columns].head(10))


In [ ]:
#@title Parse and validate timestamp correctly
df = raw.copy()

df = df.rename(columns={
    "created_at": "timestamp",
    "lat": "latitude",
    "lon": "longitude",
    "navstatus": "nav_status"
})

df["timestamp_raw"] = df["timestamp"]

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

for col in ["mmsi", "latitude", "longitude", "sog", "cog", "nav_status"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Pemeriksaan keras agar kesalahan 1970 tidak terulang
valid_year = df["timestamp"].dt.year.between(2025, 2027)
timestamp_valid_ratio = df["timestamp"].notna().mean()
year_valid_ratio = valid_year.mean()

print("Periode timestamp hasil parsing:")
print(df["timestamp"].min(), "sampai", df["timestamp"].max())
print("Timestamp valid:", f"{timestamp_valid_ratio:.2%}")
print("Tahun 2025–2027:", f"{year_valid_ratio:.2%}")

if timestamp_valid_ratio < 0.95:
    raise ValueError("Lebih dari 5% timestamp gagal dibaca.")

if year_valid_ratio < 0.95:
    raise ValueError(
        "Hasil parsing waktu tidak masuk akal. "
        "Notebook dihentikan agar tidak menghasilkan periode 1970."
    )

display(
    df[
        ["timestamp_raw", "timestamp", "mmsi",
         "latitude", "longitude", "sog", "cog"]
    ].head(10)
)


In [ ]:
#@title Load validation parameters
if (CONFIG / "pipeline_parameters.csv").exists():
    params = (
        pd.read_csv(CONFIG / "pipeline_parameters.csv")
        .set_index("parameter")["value"]
        .to_dict()
    )
else:
    params = {}

MAX_SOG_KN = float(params.get("max_sog_kn", 15))
CORRIDOR_MIN_LAT = float(params.get("corridor_min_lat", 1.34))
CORRIDOR_MAX_LAT = float(params.get("corridor_max_lat", 1.49))
CORRIDOR_MIN_LON = float(params.get("corridor_min_lon", 102.10))
CORRIDOR_MAX_LON = float(params.get("corridor_max_lon", 102.18))

print("Batas validasi:")
print("SOG maksimum:", MAX_SOG_KN, "kn")
print("Latitude:", CORRIDOR_MIN_LAT, "sampai", CORRIDOR_MAX_LAT)
print("Longitude:", CORRIDOR_MIN_LON, "sampai", CORRIDOR_MAX_LON)


In [ ]:
#@title Clean AIS and record rejection reasons
df["inside_corridor_bbox"] = (
    df["latitude"].between(CORRIDOR_MIN_LAT, CORRIDOR_MAX_LAT)
    &
    df["longitude"].between(CORRIDOR_MIN_LON, CORRIDOR_MAX_LON)
)

df["rejection_reason"] = ""

def set_reason(mask, reason):
    available = mask & df["rejection_reason"].eq("")
    df.loc[available, "rejection_reason"] = reason

set_reason(df["timestamp"].isna(), "invalid_timestamp")
set_reason(df["mmsi"].isna(), "invalid_mmsi")
set_reason(
    df["latitude"].isna() | df["longitude"].isna(),
    "missing_coordinate"
)
set_reason(
    df["latitude"].eq(0) & df["longitude"].eq(0),
    "zero_coordinate"
)
set_reason(df["valid"].ne(True), "source_flag_invalid")
set_reason(
    ~df["latitude"].between(-90, 90)
    |
    ~df["longitude"].between(-180, 180),
    "invalid_coordinate_range"
)
set_reason(
    df["sog"].lt(0) | df["sog"].gt(MAX_SOG_KN),
    "invalid_sog"
)
set_reason(
    ~df["inside_corridor_bbox"],
    "outside_research_corridor"
)

duplicate_mask = df.duplicated(
    subset=[
        "timestamp", "mmsi", "latitude",
        "longitude", "sog", "cog"
    ],
    keep="first"
)
set_reason(duplicate_mask, "duplicate")

rejected = df[df["rejection_reason"].ne("")].copy()

clean = (
    df[df["rejection_reason"].eq("")]
    .copy()
    .sort_values(["mmsi", "timestamp"])
    .reset_index(drop=True)
)

clean["time_gap_min"] = (
    clean.groupby("mmsi")["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

print("Raw:", len(df))
print("Accepted:", len(clean))
print("Rejected:", len(rejected))
print("Acceptance rate:", f"{len(clean)/max(len(df),1):.2%}")

display(
    rejected["rejection_reason"]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="rows")
)


In [ ]:
#@title Generate Stage 1 summaries
summary = pd.DataFrame({
    "metric": [
        "raw_rows",
        "accepted_rows",
        "rejected_rows",
        "acceptance_rate",
        "vessel_count",
        "start_time",
        "end_time"
    ],
    "value": [
        len(df),
        len(clean),
        len(rejected),
        len(clean) / max(len(df), 1),
        clean["mmsi"].nunique(),
        clean["timestamp"].min(),
        clean["timestamp"].max()
    ]
})

rejection_summary = (
    rejected.groupby("rejection_reason")
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)

vessel_summary = (
    clean.groupby("mmsi")
    .agg(
        messages=("mmsi", "size"),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        median_sog_kn=("sog", "median"),
        p95_sog_kn=("sog", lambda x: x.quantile(0.95)),
        maximum_sog_kn=("sog", "max"),
        median_gap_min=("time_gap_min", "median"),
        p95_gap_min=("time_gap_min", lambda x: x.quantile(0.95)),
        maximum_gap_min=("time_gap_min", "max"),
        minimum_latitude=("latitude", "min"),
        maximum_latitude=("latitude", "max"),
        minimum_longitude=("longitude", "min"),
        maximum_longitude=("longitude", "max")
    )
    .reset_index()
)

spatial_summary = pd.DataFrame({
    "metric": [
        "raw_inside_corridor",
        "raw_outside_corridor",
        "accepted_inside_corridor",
        "minimum_latitude_clean",
        "maximum_latitude_clean",
        "minimum_longitude_clean",
        "maximum_longitude_clean"
    ],
    "value": [
        int(df["inside_corridor_bbox"].sum()),
        int((~df["inside_corridor_bbox"]).sum()),
        len(clean),
        clean["latitude"].min(),
        clean["latitude"].max(),
        clean["longitude"].min(),
        clean["longitude"].max()
    ]
})

display(summary)
display(vessel_summary)
display(spatial_summary)


In [ ]:
#@title Construct time-ordered trajectory segments
def haversine_nm(lat1, lon1, lat2, lon2):
    radius_nm = 3440.065

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        +
        np.cos(lat1)
        *
        np.cos(lat2)
        *
        np.sin(dlon / 2) ** 2
    )

    return 2 * radius_nm * np.arcsin(np.sqrt(a))


traj = clean.copy()

traj["previous_timestamp"] = (
    traj.groupby("mmsi")["timestamp"].shift(1)
)
traj["previous_latitude"] = (
    traj.groupby("mmsi")["latitude"].shift(1)
)
traj["previous_longitude"] = (
    traj.groupby("mmsi")["longitude"].shift(1)
)

traj["connection_gap_min"] = (
    traj["timestamp"] - traj["previous_timestamp"]
).dt.total_seconds() / 60

traj["connection_distance_nm"] = haversine_nm(
    traj["previous_latitude"],
    traj["previous_longitude"],
    traj["latitude"],
    traj["longitude"]
)

traj["implied_speed_kn"] = (
    traj["connection_distance_nm"]
    /
    (traj["connection_gap_min"] / 60)
)

MAX_CONNECTION_GAP_MIN = 20
MAX_IMPLIED_SPEED_KN = 15

traj["same_date"] = (
    traj["timestamp"].dt.date
    ==
    traj["previous_timestamp"].dt.date
)

traj["valid_connection"] = (
    traj["same_date"]
    &
    traj["connection_gap_min"].gt(0)
    &
    traj["connection_gap_min"].le(MAX_CONNECTION_GAP_MIN)
    &
    traj["implied_speed_kn"].le(MAX_IMPLIED_SPEED_KN)
)

traj["new_segment"] = ~traj["valid_connection"]

traj["trajectory_segment_id"] = (
    traj.groupby("mmsi")["new_segment"]
    .cumsum()
    .astype(int)
)

segment_summary = (
    traj.groupby(["mmsi", "trajectory_segment_id"])
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        point_count=("timestamp", "size"),
        start_latitude=("latitude", "first"),
        start_longitude=("longitude", "first"),
        end_latitude=("latitude", "last"),
        end_longitude=("longitude", "last"),
        maximum_gap_min=("connection_gap_min", "max"),
        maximum_implied_speed_kn=("implied_speed_kn", "max")
    )
    .reset_index()
)

segment_summary["duration_min"] = (
    segment_summary["end_time"]
    -
    segment_summary["start_time"]
).dt.total_seconds() / 60

valid_segment_summary = segment_summary[
    segment_summary["point_count"].ge(2)
].copy()

print("Jumlah segmen lintasan dengan minimal 2 titik:",
      len(valid_segment_summary))

display(valid_segment_summary.head(20))


In [ ]:
#@title Save all Stage 1 CSV outputs
clean.to_csv(STAGE / "01_ais_clean.csv", index=False)
rejected.to_csv(STAGE / "01_ais_rejected.csv", index=False)
summary.to_csv(STAGE / "01_cleaning_summary.csv", index=False)
rejection_summary.to_csv(
    STAGE / "01_rejection_reason_summary.csv",
    index=False
)
vessel_summary.to_csv(
    STAGE / "01_vessel_summary.csv",
    index=False
)
spatial_summary.to_csv(
    STAGE / "01_spatial_validation_summary.csv",
    index=False
)
traj.to_csv(
    STAGE / "01_ais_with_trajectory_fields.csv",
    index=False
)
valid_segment_summary.to_csv(
    STAGE / "01_trajectory_segment_summary.csv",
    index=False
)

print("CSV Stage 1 disimpan di:", STAGE)


In [ ]:
#@title Prepare mapping libraries and vessel colors
try:
    import folium
    from folium.plugins import TimestampedGeoJson
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "folium", "ipywidgets", "-q"
    ])
    import folium
    from folium.plugins import TimestampedGeoJson
    import ipywidgets as widgets
    from IPython.display import display, clear_output

if (CONFIG / "terminal_berths.csv").exists():
    berths = pd.read_csv(CONFIG / "terminal_berths.csv")
else:
    berths = pd.DataFrame([
        ["BENGKALIS", "BENGKALIS_BERTH_1",
         1.449921, 102.136747, 0.12],
        ["BENGKALIS", "BENGKALIS_BERTH_2",
         1.449491, 102.138242, 0.12],
        ["PAKNING", "PAKNING_BERTH_1",
         1.380001, 102.148561, 0.12],
        ["PAKNING", "PAKNING_BERTH_2",
         1.378790, 102.149836, 0.12]
    ], columns=[
        "port_id", "berth_id", "latitude",
        "longitude", "occupancy_radius_nm"
    ])

COLOR_POOL = [
    "blue", "orange", "purple", "cadetblue",
    "darkred", "darkgreen", "pink", "lightblue",
    "lightgreen", "black", "darkpurple", "gray"
]

UNIQUE_MMSI = sorted(
    traj["mmsi"].dropna().astype(int).unique().tolist()
)

COLOR_MAP = {
    mmsi: COLOR_POOL[i % len(COLOR_POOL)]
    for i, mmsi in enumerate(UNIQUE_MMSI)
}

print("MMSI dan warna:")
display(pd.DataFrame({
    "mmsi": UNIQUE_MMSI,
    "color": [COLOR_MAP[x] for x in UNIQUE_MMSI]
}))


In [ ]:
#@title Define reliable visual inspection map
def build_validation_map(
    source_df,
    selected_mmsi=None,
    selected_date=None,
    start_hour=0,
    end_hour=24,
    include_points=True,
    include_trajectories=True,
    max_points=6000,
    output_name="01_ais_selected_validation_map.html"
):
    view = source_df.copy()

    if selected_mmsi not in [None, "ALL"]:
        view = view[view["mmsi"].astype(int).eq(int(selected_mmsi))]

    if selected_date not in [None, "ALL"]:
        selected_date = pd.to_datetime(selected_date).date()
        view = view[view["timestamp"].dt.date.eq(selected_date)]

    hour_decimal = (
        view["timestamp"].dt.hour
        +
        view["timestamp"].dt.minute / 60
        +
        view["timestamp"].dt.second / 3600
    )

    view = view[
        hour_decimal.ge(float(start_hour))
        &
        hour_decimal.lt(float(end_hour))
    ].copy()

    if view.empty:
        raise ValueError(
            "Tidak ada data pada kombinasi MMSI, tanggal, dan jam tersebut."
        )

    center_lat = view["latitude"].median()
    center_lon = view["longitude"].median()

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles="OpenStreetMap",
        control_scale=True
    )

    corridor_layer = folium.FeatureGroup(
        name="Research corridor",
        show=True
    )

    bbox = [
        [CORRIDOR_MIN_LAT, CORRIDOR_MIN_LON],
        [CORRIDOR_MIN_LAT, CORRIDOR_MAX_LON],
        [CORRIDOR_MAX_LAT, CORRIDOR_MAX_LON],
        [CORRIDOR_MAX_LAT, CORRIDOR_MIN_LON],
        [CORRIDOR_MIN_LAT, CORRIDOR_MIN_LON]
    ]

    folium.PolyLine(
        bbox,
        color="red",
        weight=3,
        dash_array="8,6",
        tooltip="Research corridor"
    ).add_to(corridor_layer)

    corridor_layer.add_to(m)

    berth_layer = folium.FeatureGroup(
        name="Berth locations",
        show=True
    )

    for _, b in berths.iterrows():
        folium.Marker(
            [b["latitude"], b["longitude"]],
            tooltip=str(b["berth_id"]),
            popup=(
                f"<b>{b['berth_id']}</b><br>"
                f"Port: {b['port_id']}"
            ),
            icon=folium.Icon(
                color="green",
                icon="anchor",
                prefix="fa"
            )
        ).add_to(berth_layer)

        folium.Circle(
            [b["latitude"], b["longitude"]],
            radius=float(b.get("occupancy_radius_nm", 0.12)) * 1852,
            color="green",
            fill=True,
            fill_opacity=0.08
        ).add_to(berth_layer)

    berth_layer.add_to(m)

    for mmsi, vessel_df in view.groupby("mmsi"):
        mmsi = int(mmsi)
        color = COLOR_MAP[mmsi]

        vessel_layer = folium.FeatureGroup(
            name=f"MMSI {mmsi}",
            show=True
        )

        if include_trajectories:
            for segment_id, segment in vessel_df.groupby(
                "trajectory_segment_id"
            ):
                segment = segment.sort_values("timestamp")

                if len(segment) < 2:
                    continue

                folium.PolyLine(
                    segment[["latitude", "longitude"]].values.tolist(),
                    color=color,
                    weight=3,
                    opacity=0.8,
                    tooltip=(
                        f"MMSI {mmsi}<br>"
                        f"Segment {int(segment_id)}<br>"
                        f"{segment['timestamp'].min()}<br>"
                        f"to {segment['timestamp'].max()}"
                    )
                ).add_to(vessel_layer)

        if include_points:
            point_df = vessel_df.sort_values("timestamp")

            if len(point_df) > max_points:
                indexes = np.linspace(
                    0, len(point_df) - 1, max_points
                ).astype(int)
                point_df = point_df.iloc[indexes]

            for _, r in point_df.iterrows():
                folium.CircleMarker(
                    [r["latitude"], r["longitude"]],
                    radius=3,
                    color=color,
                    weight=1,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.8,
                    tooltip=(
                        f"MMSI {mmsi} | "
                        f"{r['timestamp']}"
                    ),
                    popup=folium.Popup(
                        (
                            f"<b>MMSI:</b> {mmsi}<br>"
                            f"<b>Timestamp:</b> {r['timestamp']}<br>"
                            f"<b>SOG:</b> {r['sog']:.2f} kn<br>"
                            f"<b>COG:</b> {r['cog']:.2f}°<br>"
                            f"<b>Gap:</b> {r['connection_gap_min']:.2f} min<br>"
                            f"<b>Implied speed:</b> "
                            f"{r['implied_speed_kn']:.2f} kn<br>"
                            f"<b>Segment:</b> "
                            f"{int(r['trajectory_segment_id'])}"
                        ),
                        max_width=360
                    )
                ).add_to(vessel_layer)

        vessel_layer.add_to(m)

    legend_items = ""

    for mmsi in sorted(view["mmsi"].astype(int).unique()):
        color = COLOR_MAP[mmsi]
        count = int(view["mmsi"].astype(int).eq(mmsi).sum())

        legend_items += f'''
        <div style="display:flex;align-items:center;margin-bottom:5px;">
          <span style="
            width:13px;height:13px;border-radius:50%;
            background:{color};border:1px solid #333;
            margin-right:7px;">
          </span>
          MMSI {mmsi} ({count:,} points)
        </div>
        '''

    legend_html = f'''
    <div style="
      position:fixed;bottom:35px;right:20px;z-index:9999;
      background:rgba(255,255,255,0.95);border:2px solid #777;
      border-radius:6px;padding:10px;font-family:Arial;
      font-size:12px;max-height:300px;overflow-y:auto;">
      <b>AIS vessel legend</b>
      <div style="margin-top:8px;">{legend_items}</div>
      <hr>
      <div>Solid vessel-coloured line: valid time-ordered trajectory</div>
      <div>Red dashed line: research corridor</div>
      <div>Green marker/circle: berth and radius</div>
    </div>
    '''

    m.get_root().html.add_child(
        folium.Element(legend_html)
    )

    folium.LayerControl(
        collapsed=False,
        position="topright"
    ).add_to(m)

    m.fit_bounds([
        [view["latitude"].min(), view["longitude"].min()],
        [view["latitude"].max(), view["longitude"].max()]
    ])

    output_path = STAGE / output_name
    m.save(str(output_path))

    print("Data terpilih:", len(view), "titik")
    print("MMSI:", sorted(view["mmsi"].astype(int).unique().tolist()))
    print("Periode:", view["timestamp"].min(), "sampai", view["timestamp"].max())
    print("Peta disimpan:", output_path)

    return m


In [ ]:
#@title Generate full Stage 1 validation map
# Peta keseluruhan untuk validasi awal
full_map = build_validation_map(
    source_df=traj,
    selected_mmsi="ALL",
    selected_date="ALL",
    start_hour=0,
    end_hour=24,
    include_points=True,
    include_trajectories=True,
    max_points=1500,
    output_name="01_ais_full_validation_map.html"
)

full_map


In [ ]:
#@title Interactive MMSI, date, and hour inspection
available_dates = sorted(
    traj["timestamp"]
    .dt.strftime("%Y-%m-%d")
    .unique()
    .tolist()
)

mmsi_selector = widgets.Dropdown(
    options=["ALL"] + [str(x) for x in UNIQUE_MMSI],
    value="ALL",
    description="MMSI:"
)

date_selector = widgets.Dropdown(
    options=["ALL"] + available_dates,
    value="ALL",
    description="Tanggal:"
)

start_hour_selector = widgets.IntSlider(
    value=0,
    min=0,
    max=23,
    step=1,
    description="Jam mulai:"
)

end_hour_selector = widgets.IntSlider(
    value=24,
    min=1,
    max=24,
    step=1,
    description="Jam akhir:"
)

show_points_selector = widgets.Checkbox(
    value=True,
    description="Tampilkan titik"
)

show_lines_selector = widgets.Checkbox(
    value=True,
    description="Tampilkan trajectory"
)

generate_button = widgets.Button(
    description="Buat peta inspeksi",
    button_style="primary"
)

inspection_output = widgets.Output()

def generate_selected_map(_):
    with inspection_output:
        clear_output(wait=True)

        if end_hour_selector.value <= start_hour_selector.value:
            print("Jam akhir harus lebih besar daripada jam mulai.")
            return

        try:
            selected_map = build_validation_map(
                source_df=traj,
                selected_mmsi=mmsi_selector.value,
                selected_date=date_selector.value,
                start_hour=start_hour_selector.value,
                end_hour=end_hour_selector.value,
                include_points=show_points_selector.value,
                include_trajectories=show_lines_selector.value,
                max_points=5000,
                output_name="01_ais_selected_validation_map.html"
            )
            display(selected_map)

        except Exception as error:
            print("Peta tidak dapat dibuat:", error)

generate_button.on_click(generate_selected_map)

display(
    widgets.VBox([
        widgets.HTML("<b>Filter visual AIS</b>"),
        mmsi_selector,
        date_selector,
        start_hour_selector,
        end_hour_selector,
        widgets.HBox([
            show_points_selector,
            show_lines_selector
        ]),
        generate_button,
        inspection_output
    ])
)


In [ ]:
#@title Stage 1 completion report
print("Stage 1 selesai.")
print("Output tersimpan di:", STAGE)

for file_name in sorted(STAGE.glob("01_*")):
    print("-", file_name.name)


## Keluaran interpretatif otomatis\n\nCSV/JSON dipertahankan untuk kontrak data. Sel berikut membuat keluaran yang dapat dibaca dan dieksplorasi tanpa membuka CSV mentah.

In [ ]:
#@title Export readable Stage 1 outputs
from src.mfar_visuals import stage1_quality_outputs
_readable_outputs = stage1_quality_outputs(
    summary, rejection_summary, vessel_summary, STAGE_01_DIR
)
print("Readable Stage 1 outputs:")
for _path in _readable_outputs:
    print("-", _path.name)


In [ ]:
#@title Execution metadata and saved-artifact report
_stage_dir = STAGE_01_DIR
_saved_files = sorted(_stage_dir.glob("01_*"))
write_execution_metadata(
    stage=1, notebook=NOTEBOOK_NAME, started_at=_MFAR_STARTED_AT,
    input_paths=[AIS_RAW_PATH, VEHICLE_ARRIVAL_PATH],
    input_rows={"raw": len(raw)},
    output_rows={"clean": len(clean)},
    output_files=_saved_files,
)
